# Hurtownia Danych Iowa Liquor Sales: Dokumentacja ETL i Architektura
Poniższy notatnik przedstawia strukturę bazy danych i przepływ danych (ETL) zrealizowany w ramach projektu. 
Zastosowano Architekturę Medalionową, która dzieli proces przetwarzania na trzy warstwy.
Warstwy w hurtowni:
1. **Bronze (Staging)** - Dane surowe, zrzucone prosto ze źródła. Tabela, do której ładujemy CSV.
2. **Silver (Data Warehouse)** - Wyczyszczone dane w Modelu Gwiazdy (tabela faktów i wymiary).
3. **Gold (Semantic Layer)** - Gotowe widoki analityczne (marty) podzielone tematycznie, do bezpośredniego odpytywania przez BI.


In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import pandas as pd
from src.utils.db import sqlserver_connection
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
def query_db(sql_query: str) -> pd.DataFrame:
    with sqlserver_connection() as conn:
        return pd.read_sql(sql_query, conn)


## 1. Warstwa Bronze (Staging) - Surowe dane
Na początku dane ładujemy do tabeli `stg.iowa_liquor_sales_raw`. Poniższe zapytanie pokazuje rekordy z błędami w źródle, które następnie musieliśmy oczyścić w procesie ETL:


In [2]:
sql_dirty = """
SELECT TOP 5 
    invoice_and_item_number,
    date,
    store_location,      -- Zapisane jako tekst POINT(X Y)
    category_name,       -- Brak kategorii (NULL)
    state_bottle_cost,   -- Znak $ i przecinki, uniemożliwiające operacje liczbowe
    sale_dollars         -- Przechowywane jako tekst
FROM stg.iowa_liquor_sales_raw
WHERE category_name IS NULL OR store_location LIKE 'POINT%';
"""
display(query_db(sql_dirty))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,invoice_and_item_number,date,store_location,category_name,state_bottle_cost,sale_dollars
0,INV-54554000001,2023-01-02,POINT (-93.61378 41.60575),100% AGAVE TEQUILA,14.50,261.00
1,INV-54554000002,2023-01-02,POINT (-93.61378 41.60575),AMERICAN VODKAS,4.65,418.80
2,INV-54554000003,2023-01-02,POINT (-93.61378 41.60575),IMPORTED FLAVORED VODKA,9.96,358.56
3,INV-54554000004,2023-01-02,POINT (-93.61378 41.60575),CREAM LIQUEURS,17.00,306.00
4,INV-54554000005,2023-01-02,POINT (-93.61378 41.60575),SPICED RUM,12.49,1124.40


### Logika ładowania (Python / Pandas)
Zanim zasilimy docelowe tabele wymiarów, w Pythonie robimy kilka transformacji:
1. Zmieniamy formaty finansowe: używamy metod wektoryzowanych Pandas (np. `.str.replace('$', '')`), aby rzutować łańcuchy znaków na liczby.
2. Wyciągamy współrzędne: wyrażeniami regularnymi parsujemy string POINT na osobne kolumny `latitude` i `longitude`.
3. Dodajemy hashowanie: generujemy unikalny klucz `source_row_hash` za pomocą algorytmu `SHA-256`, ułatwiający deduplikację rekordów.
4. Otwieramy połączenie ODBC i używamy `fast_executemany = True` do szybszego zapisu w bazie (batch insert).


## 2. Warstwa Silver (Data Warehouse) - SQL i Model Gwiazdy
Część zapytań transformujących (T z ETL) wykonujemy bezpośrednio w SQL na silniku bazy danych.
### Deduplikacja wymiarów
Źródło często dubluje np. sklepy, mając różne daty dla tego samego ID sklepu. Przy zapisie do `dw.dim_store` używamy funkcji okna (Window Functions):
```sql
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY COALESCE(NULLIF(store_number, ''), 'UNKNOWN')
        ORDER BY date DESC, staging_key DESC
    ) AS rn
    FROM stg.iowa_liquor_sales_raw
)
INSERT INTO dw.dim_store ... SELECT ... FROM ranked WHERE rn = 1;
```
Warunek `rn = 1` zapewnia, że w wymiarze ląduje jeden, najświeższy wpis dla danego sklepu.
### Radzenie sobie z pustymi wartościami
Tabela faktów nie powinna mieć pustych kluczy obcych. Jeśli brakuje kategorii na wejściu, klauzula `COALESCE(NULLIF(category, ''), 'UNKNOWN')` przypisuje klucz "UNKNOWN". W tabelach wymiarów (np. `dim_category`) znajduje się sztucznie wygenerowany wiersz odpowiadający "UNKNOWN", aby móc zawsze złączyć dane (JOIN).


In [3]:
print("Przykładowe, wyczyszczone lokalizacje ze stg do dim_store:")
display(query_db("SELECT TOP 3 store_key, store_name, latitude, longitude FROM dw.dim_store WHERE latitude IS NOT NULL;"))
print("\nRekord techniczny 'UNKNOWN' w wymiarach:")
display(query_db("SELECT * FROM dw.dim_category WHERE category_number = 'UNKNOWN';"))


Przykładowe, wyczyszczone lokalizacje ze stg do dim_store:


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_key,store_name,latitude,longitude
0,1,JACK & JILL STORE / WEST BRANCH,41.670494,-91.343413
1,2,LOCAL LIQUOR / PANORA,41.692594,-94.357208
2,3,LEGENDARY RYE / BAD BEAR ENTERPRISES (ET),42.067186,-94.866874



Rekord techniczny 'UNKNOWN' w wymiarach:


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_key,category_number,category_name


### Tabela Faktów
Dzięki `dw.fact_sales` złączamy wyczyszczone identyfikatory. Bezpośrednio przy ładownaniu dodana jest wyliczona kolumna `margin_amount` określająca kwotę marży na podstawie różnicy kosztu i ceny.


In [4]:
print("Fragment tabeli faktów z kolumną margin_amount:")
display(query_db("SELECT TOP 5 invoice_number, store_key, category_key, sale_dollars, state_bottle_cost, margin_amount FROM dw.fact_sales;"))


Fragment tabeli faktów z kolumną margin_amount:


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,invoice_number,store_key,category_key,sale_dollars,state_bottle_cost,margin_amount
0,INV-54554000001,984,2,261.00,14.50,87.00
1,INV-54554000002,984,30,418.80,4.65,139.80
2,INV-54554000003,984,23,358.56,9.96,119.52
3,INV-54554000004,984,26,306.00,17.00,102.00
4,INV-54554000005,984,46,1124.40,12.49,375.00


## 3. Warstwa Gold (Semantic Layer) - 16 Widoków
Aby analityk nie musiał pisać skomplikowanych złączeń, warstwa semantyczna udostępnia 16 gotowych widoków zgrupowanych tematycznie. Liczą one wybrane agregacje po stronie bazy danych. Poniżej przedstawiono przykładowe zapytania z każdego widoku.


### Grupa 1: Główne statystyki (High-Level / KPI)
#### 1. `sem.vw_kpi_summary`
Ogólne podsumowanie finansowe.


In [5]:
display(query_db("SELECT * FROM sem.vw_kpi_summary;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,total_sales,total_margin,sales_line_count,total_bottles_sold,total_volume_liters,invoice_count,store_count,product_count,category_count,vendor_count,avg_invoice_value,avg_bottles_per_invoice,avg_margin_percent,sales_per_store,sales_per_liter
0,4.466417e+08,1.492504e+08,2635879,31302201.0,23756277.41,2635879,2110,5231,48,250,169.446953,11.875431,33.42,211678.514578,18.800995


#### 2. `sem.vw_etl_status`
Liczba wierszy we wszystkich tabelach modelu, służąca do weryfikacji po stronie operacyjnej.


In [6]:
display(query_db("SELECT * FROM sem.vw_etl_status;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,status_generated_at,staging_row_count,fact_row_count,dim_date_count,dim_store_count,dim_product_count,dim_category_count,dim_vendor_count,dim_packaging_count,min_date,max_date,last_staging_load_timestamp,last_fact_load_timestamp
0,2026-07-04 21:41:10.100,2639557,2635879,290,2110,5231,48,250,69,2023-01-02,2023-12-30,2026-07-04 19:53:26.060703,2026-07-04 19:53:43.247454


### Grupa 2: Czas
#### 3. `sem.vw_sales_overview`
Szczegółowy widok ze wszystkimi powiązanymi wymiarami dla tabeli faktów.


In [7]:
display(query_db("SELECT TOP 5 date, invoice_number, store_name, category_name, sale_dollars FROM sem.vw_sales_overview;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,date,invoice_number,store_name,category_name,sale_dollars
0,2023-01-02,INV-54554000001,CENTRAL CITY 2,100% AGAVE TEQUILA,261.00
1,2023-01-02,INV-54554000002,CENTRAL CITY 2,AMERICAN VODKAS,418.80
2,2023-01-02,INV-54554000003,CENTRAL CITY 2,IMPORTED FLAVORED VODKA,358.56
3,2023-01-02,INV-54554000004,CENTRAL CITY 2,CREAM LIQUEURS,306.00
4,2023-01-02,INV-54554000005,CENTRAL CITY 2,SPICED RUM,1124.40


#### 4. `sem.vw_sales_by_day_type`
Agregacja sprzedazy z uwzględnieniem podziału na weekend / dzień roboczy z tabeli `dim_date`.


In [8]:
display(query_db("SELECT * FROM sem.vw_sales_by_day_type;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,is_weekend,day_type,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count
0,2023,3,7,2023-07,True,Weekend,1186208.53,87163.0,61733.02,395977.14,8854,8854
1,2023,1,1,2023-01,True,Weekend,838436.05,66807.0,45927.71,279843.16,7061,7061
2,2023,2,6,2023-06,True,Weekend,1424375.15,105970.0,76931.37,474511.15,9701,9701
3,2023,3,9,2023-09,True,Weekend,1728305.20,128850.0,95695.29,577184.04,12263,12263
4,2023,3,8,2023-08,True,Weekend,298.45,12.0,15.00,102.50,7,7
5,2023,4,11,2023-11,True,Weekend,3412682.64,271801.0,178442.29,1138208.97,20659,20659
6,2023,1,3,2023-03,False,Weekday,36433363.96,2632359.0,2016057.37,12183998.11,221299,221299
7,2023,2,5,2023-05,False,Weekday,39704733.36,2784835.0,2186163.82,13392595.99,232960,232960
8,2023,4,11,2023-11,False,Weekday,35929764.34,2395754.0,1884156.66,11984428.10,202653,202653
9,2023,4,12,2023-12,True,Weekend,3407799.86,262935.0,170366.75,1136876.98,24738,24738


#### 5. `sem.vw_sales_by_month`
Podsumowanie dla roku i miesiąca.


In [9]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_month ORDER BY year, month;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,store_count
0,2023,1,1,2023-01,32582340.63,2358449.0,1747102.70,10888433.83,212850,212850,1857
1,2023,1,2,2023-02,32134462.65,2289374.0,1771119.28,10750429.35,189294,189294,1824
2,2023,1,3,2023-03,36436060.72,2632521.0,2016226.87,12184903.75,221313,221313,1859
3,2023,2,4,2023-04,32915910.22,2393665.0,1793682.48,10986201.79,198446,198446,1832
4,2023,2,5,2023-05,39721449.29,2786013.0,2187271.06,13398186.64,233060,233060,1879


#### 6. `sem.vw_avg_sales_per_store_by_month_region`
Średnia wartość sprzedaży na sklep zgrupowana po dacie oraz regionie.


In [10]:
display(query_db("SELECT TOP 5 * FROM sem.vw_avg_sales_per_store_by_month_region;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,state_name,county,city,store_count,avg_sales_per_store,avg_bottles_per_store,avg_volume_liters_per_store,avg_margin_per_store
0,2023,2,4,2023-04,Iowa,Scott,Eldridge,6,17428.365000,1322.000000,893.505000,5815.576666
1,2023,3,9,2023-09,Iowa,Cerro Gordo,Mason City,17,26915.348235,2127.411764,1585.295294,8976.100588
2,2023,1,3,2023-03,Iowa,Cass,Atlantic,7,14203.137142,955.714285,797.464285,4753.011428
3,2023,1,1,2023-01,Iowa,Linn,Marion,19,13313.450000,1071.684210,795.760526,4448.538421
4,2023,4,12,2023-12,Iowa,Fremont,Hamburg,1,5829.400000,557.000000,241.640000,1943.600000


### Grupa 3: Produkty i marża
#### 7. `sem.vw_sales_by_category`
Zestawienie kategorii. Posiada pre-kalkulowany udział wartości kategorii w całości biznesu (kolumna `sales_share_percent`).


In [11]:
display(query_db("SELECT TOP 5 category_name, total_sales, sales_share_percent, avg_margin_per_bottle FROM sem.vw_sales_by_category ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_name,total_sales,sales_share_percent,avg_margin_per_bottle
0,AMERICAN VODKAS,67412480.47,15.09,3.453703
1,CANADIAN WHISKIES,50411669.32,11.29,5.498948
2,STRAIGHT BOURBON WHISKIES,38747595.32,8.68,7.536290
3,100% AGAVE TEQUILA,32384034.97,7.25,9.742586
4,WHISKEY LIQUEUR,26564898.28,5.95,2.004608


#### 8. `sem.vw_category_sales_over_time`
Sprzedaż kategorii pogrupowana po czasie (Rok/Miesiąc).


In [12]:
display(query_db("SELECT TOP 5 * FROM sem.vw_category_sales_over_time;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,year,quarter,month,year_month,category_name,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,store_count
0,2023,1,2,2023-02,IMPORTED BRANDIES,940218.52,40581.0,18497.41,313499.29,3378,3378,895
1,2023,2,4,2023-04,FLAVORED RUM,737310.49,47459.0,43605.20,245864.15,4507,4507,995
2,2023,4,12,2023-12,FLAVORED GIN,109547.01,4204.0,3168.55,36517.38,416,416,196
3,2023,2,4,2023-04,BOTTLED IN BOND BOURBON,111021.60,3330.0,2746.50,37007.62,710,710,354
4,2023,1,1,2023-01,TRIPLE SEC,62251.55,14834.0,14623.00,20776.54,639,639,306


#### 9. `sem.vw_top_products`
Ranking najlepiej sprzedających się artykułów.


In [13]:
display(query_db("SELECT TOP 5 item_description, total_bottles_sold, total_sales FROM sem.vw_top_products ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,item_description,total_bottles_sold,total_sales
0,TITOS HANDMADE VODKA,400398.0,11411343.00
1,TITOS HANDMADE VODKA,506485.0,10008143.60
2,BLACK VELVET,524325.0,8567077.98
3,CAPTAIN MORGAN ORIGINAL SPICED BARREL,256199.0,7063866.82
4,TITOS HANDMADE VODKA,438152.0,6572280.00


#### 10. `sem.vw_margin_analysis`
Analiza marży na dany produkt dla każdego dostawcy.


In [14]:
display(query_db("SELECT TOP 5 category_name, vendor_name, avg_unit_margin, total_margin FROM sem.vw_margin_analysis ORDER BY total_margin DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,category_name,vendor_name,avg_unit_margin,total_margin
0,AMERICAN VODKAS,FIFTH GENERATION INC,5.734500,10027926.96
1,CANADIAN WHISKIES,HEAVEN HILL BRANDS,3.629779,4119795.00
2,WHISKEY LIQUEUR,SAZERAC COMPANY INC,2.815516,3879317.94
3,CANADIAN WHISKIES,DIAGEO AMERICAS,8.952332,3410927.00
4,TENNESSEE WHISKIES,BROWN FORMAN CORP.,9.499180,3211062.02


#### 11. `sem.vw_sales_by_packaging`
Wolumen opakowań rozbity na rozmiar butelki.


In [15]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_packaging ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,pack,bottle_volume_ml,volume_group,total_sales,total_bottles_sold,total_volume_liters,total_margin,sales_line_count,invoice_count,sales_share_percent
0,12,750,standard,1.316039e+08,8054198.0,6040648.50,43918698.82,828691,828691,29.47
1,6,1750,extra_large,1.020161e+08,5198080.0,9096640.00,34084839.23,457790,457790,22.84
2,12,1000,large,7.177420e+07,4537105.0,4537105.00,23966457.47,226615,226615,16.07
3,6,750,standard,6.171046e+07,1968954.0,1476715.50,20602243.01,332328,332328,13.82
4,24,375,small,2.056612e+07,3411086.0,1279015.35,6867397.04,201109,201109,4.60


#### 12. `sem.vw_sales_by_vendor`
Statystyki dostawców dostarczających towar.


In [16]:
display(query_db("SELECT TOP 5 * FROM sem.vw_sales_by_vendor ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,vendor_name,total_sales,total_bottles_sold,total_margin,sales_line_count,sales_share_percent
0,DIAGEO AMERICAS,89105882.60,4237650.0,29748794.14,393222,19.95
1,SAZERAC COMPANY INC,66845842.81,8381527.0,22446497.38,454451,14.97
2,FIFTH GENERATION INC,30880585.64,1672757.0,10295491.76,88069,6.91
3,JIM BEAM BRANDS,30077014.54,1879448.0,10034039.49,205949,6.73
4,PERNOD RICARD USA,27732294.21,1369487.0,9252738.93,133186,6.21


### Grupa 4: Geografia
#### 13. `sem.vw_sales_by_geography`
Agregacja wg terytorium - hrabstwo oraz miasto.


In [17]:
display(query_db("SELECT TOP 5 county, city, total_sales, total_bottles_sold FROM sem.vw_sales_by_geography ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,county,city,total_sales,total_bottles_sold
0,Polk,Des Moines,54716593.67,3964525.0
1,Linn,Cedar Rapids,28165657.82,2047863.0
2,Scott,Davenport,20903373.82,1700718.0
3,Pottawattamie,Council Bluffs,15282146.05,1140320.0
4,Woodbury,Sioux City,14566939.00,1065968.0


#### 14. `sem.vw_sales_map_points`
Współrzędne (szerokość i długość geograficzna) sklepów wykorzystywane do nanoszenia lokalizacji na mapy w narzędziach BI.


In [18]:
display(query_db("SELECT TOP 5 store_name, city, latitude, longitude, total_sales FROM sem.vw_sales_map_points ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_name,city,latitude,longitude,total_sales
0,HY-VEE #3 / BDI / DES MOINES,Des Moines,41.554269,-93.594781,15275187.46
1,CENTRAL CITY 2,Des Moines,41.605835,-93.613286,13740361.39
2,ANOTHER ROUND / DEWITT,Dewitt,41.809631,-90.538996,6894898.17
3,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,Iowa City,41.642516,-91.529426,6112597.82
4,BENZ DISTRIBUTING,Cedar Rapids,41.975513,-91.659640,5235975.06


#### 15. `sem.vw_sales_by_store`
Podsumowanie po sklepie - marże i przychody.


In [19]:
display(query_db("SELECT TOP 5 store_name, total_sales, total_margin, invoice_count FROM sem.vw_sales_by_store ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,store_name,total_sales,total_margin,invoice_count
0,HY-VEE #3 / BDI / DES MOINES,15275187.46,5098821.19,20611
1,CENTRAL CITY 2,13740361.39,4586639.30,20506
2,ANOTHER ROUND / DEWITT,6894898.17,2301835.13,11761
3,HY-VEE WINE AND SPIRITS #1 (1281) / IOWA CITY,6112597.82,2040235.09,10041
4,BENZ DISTRIBUTING,5235975.06,1748162.77,14760


#### 16. `sem.vw_volume_vs_revenue`
Korelacja przewiezionego wolumenu (w litrach) względem uzyskanej wartości ze sprzedaży na dany dystrykt.


In [20]:
display(query_db("SELECT TOP 5 city, total_volume_liters, total_sales, sales_per_liter FROM sem.vw_volume_vs_revenue ORDER BY total_sales DESC;"))


C:\Users\paula\AppData\Local\Temp\ipykernel_22640\1681752413.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql_query, conn)


,city,total_volume_liters,total_sales,sales_per_liter
0,Des Moines,2628116.69,54716593.67,20.819697
1,Cedar Rapids,1479865.42,28165657.82,19.032580
2,Davenport,1119286.67,20903373.82,18.675621
3,Council Bluffs,772569.21,15282146.05,19.780941
4,Sioux City,760230.63,14566939.00,19.161210
